# RO-231 Solving trajectory optimisation: every number in the Note

The robot, the room, the straight drive, single and multiple shooting, trapezoidal collocation, iLQR and plain
gradient descent live in `images/trajkit.py` (a copy of the one in RO-058's `images/`); which unknowns each equation
touches is counted in `images/coupling.py`. Every solve starts from a fixed first guess, so nothing is random and
there is no seed to set. The whole notebook takes a few minutes the first time (the solves are then kept in a
temporary folder).


In [1]:
import sys
sys.path.insert(0, 'images')
import numpy as np
import trajkit as tk
import coupling
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
S = tk.solutions()


## 1. Single shooting by hand: the straight drive (Section 2.2, Figure 2)


In [2]:
for a, end, miss, grad in tk.straight_gd(steps=6):
    print(f'a = {a:.6f}  end = {end:.6f}  miss = {miss:.6f}  slope = {grad:.4f}')
print('the a that makes the miss 0:', 3 / 25, 'm/s^2; speed at 5 s:', r3(5 * 3 / 25), 'm/s')
print('two halves: end = 37.5 a1 + 12.5 a2:', 0.5 * 5 ** 2 + 5 * 5, 0.5 * 5 ** 2)


a = 0.000000  end = 0.000000  miss = -3.000000  slope = -150.0000
a = 0.090000  end = 2.250000  miss = -0.750000  slope = -37.5000
a = 0.112500  end = 2.812500  miss = -0.187500  slope = -9.3750
a = 0.118125  end = 2.953125  miss = -0.046875  slope = -2.3438
a = 0.119531  end = 2.988281  miss = -0.011719  slope = -0.5859
a = 0.119883  end = 2.997070  miss = -0.002930  slope = -0.1465
the a that makes the miss 0: 0.12 m/s^2; speed at 5 s: 0.6 m/s
two halves: end = 37.5 a1 + 12.5 a2: 37.5 12.5


## 2. The 8 s drive: single and multiple shooting (Sections 2.1 to 2.4, Figures 1, 3, 4)


In [3]:
print('interval', r3(8 / 30), 's; RK4 step', r3(8 / 30 / 4), 's')
U = S['single']['U']
print('acceleration of the best drive: first', r2(U[0, 0]), 'last', r2(U[-1, 0]), 'm/s^2')
for name in ('single', 'multi', 'colloc'):
    s = S[name]
    print(f"{name:7s} iterations {len(s['hist']) - 1:3d}  effort {s['effort']:.4f}  largest defect {np.abs(s['defects']).max():.1e}")
print('single shooting: least gap to the table along the drive', r4(tk.clearance(S['single']['traj'][:, 0], S['single']['traj'][:, 1]).min()), 'm')
shot = tk.simulate(tk.START, tk.first_shot(30, 8.0), 8.0)
print('first shot misses by', r3(np.hypot(*(shot[-1, :2] - tk.GOAL[:2]))), 'm')
segs = S['multi']['hist'][0][0]
print('multiple shooting, first gaps', [r3(np.hypot(*(segs[j][-1, :2] - segs[j + 1][0, :2]))) for j in range(4)])
eps = 1e-5
end = lambda U: tk.simulate(tk.START, U, 8.0)[-1, :2]
base = end(U)
for k in (0, 29):
    Up = U.copy(); Up[k, 1] += eps
    print(f'turn rate in interval {k + 1}: end moves', r4(np.hypot(*(end(Up) - base)) / eps * 0.1), 'm per 0.1 rad/s')
Um = S['multi']['U'].copy(); Up = Um.copy(); Up[0, 1] += eps
e0 = tk.simulate(tk.START, Um[:6], 8.0 / 5)[-1, :2]; e1 = tk.simulate(tk.START, Up[:6], 8.0 / 5)[-1, :2]
print('multiple shooting: the first turn rate moves the end of its 1.6 s piece', r4(np.hypot(*(e1 - e0)) / eps * 0.1), 'm per 0.1 rad/s')
print('unknowns: single', U.size, '; multiple', Um.size + S['multi']['S'][1:].size, '; collocation', S['colloc']['X'].size + S['colloc']['U'].size)


interval 0.267 s; RK4 step 0.067 s
acceleration of the best drive: first 0.27 last -0.27 m/s^2
single  iterations  73  effort 0.2171  largest defect 2.7e-07
multi   iterations  81  effort 0.2171  largest defect 2.6e-07
colloc  iterations  72  effort 0.2178  largest defect 2.8e-07
single shooting: least gap to the table along the drive -0.001 m
first shot misses by 1.048 m
multiple shooting, first gaps [0.59, 0.405, 0.077, 0.284]
turn rate in interval 1: end moves 0.0791 m per 0.1 rad/s
turn rate in interval 30: end moves 0.0001 m per 0.1 rad/s
multiple shooting: the first turn rate moves the end of its 1.6 s piece 0.0081 m per 0.1 rad/s
unknowns: single 60 ; multiple 76 ; collocation 186


## 3. Collocation: the straight drive and the 8 s drive (Sections 3.1, 3.2, Figures 5, 6)


In [4]:
print('exact least time of the straight drive', r3(tk.bang_coast_bang()), 's')
for N in (10, 20, 40):
    print(N, 'intervals: least time', r4(tk.straight_collocation(N=N)['T']), 's; unknowns', (N + 1) * 3 + 1)
X0, U0, _ = S['colloc']['hist'][0]; Fd = tk.f(X0, U0); pred = X0[:-1] + 8 / 30 / 2 * (Fd[:-1] + Fd[1:])
print('collocation, first largest defect', r3(np.hypot(*(pred[:, :2] - X0[1:, :2]).T).max()), 'm')


exact least time of the straight drive 6.2 s
10 intervals: least time 6.3643 s; unknowns 34
20 intervals: least time 6.2386 s; unknowns 64
40 intervals: least time 6.2086 s; unknowns 124
collocation, first largest defect 0.218 m


## 4. Which unknowns each equation touches (Section 3.3, Figure 7)


In [5]:
P = coupling.patterns()
for k, A in P.items():
    print(f'{k:7s} {A.shape[0]:3d} equations x {A.shape[1]:3d} unknowns: {100 * A.mean():.1f} percent non-zero')
print('single shooting, unknowns per equation of the miss:', P['single'].sum(axis=1).tolist())
print('multiple shooting, unknowns per join (4 equations):', [int(P['multi'][4 * j:4 * j + 4].any(axis=0).sum()) for j in range(5)])
print('collocation, unknowns per collocation constraint (4 equations), largest:', max(int(P['colloc'][4 + 4 * k:8 + 4 * k].any(axis=0).sum()) for k in range(30)))


single    4 equations x  60 unknowns: 75.0 percent non-zero
multi    20 equations x  76 unknowns: 15.0 percent non-zero
colloc  128 equations x 186 unknowns: 2.5 percent non-zero
single shooting, unknowns per equation of the miss: [60, 60, 30, 30]
multiple shooting, unknowns per join (4 equations): [16, 20, 20, 20, 16]
collocation, unknowns per collocation constraint (4 equations), largest: 12


## 5. iLQR and gradient descent (Overview, Section 4, Figure 8)


In [6]:
rs = tk.ilqr_straight()
print('straight drive iLQR costs', [round(c, 5) for c in rs['costs']], '; first control', r4(rs['us'][0]))
print('exact least effort for 3 m in 10 s: 12 d^2 / T^3 =', 12 * 9 / 1000, '; a at t = h/2:', r4(0.18 * (1 - 0.2 * (10 / 60))))
il = S['ilqr']; ci = [h[2] for h in il['hist']]
print('iLQR iterations', len(ci) - 1, '; cost after 0 and 13:', r4(ci[0]), r4(ci[13]), '; final', r4(ci[-1]),
      '; 13 against final:', r3(ci[13] / ci[-1]))
gd = S['gd']
print('gradient descent after 13 and 1000 iterations:', [round(gd['costs'][i], 1) for i in (13, 1000)],
      '; at 13 against iLQR at 13:', round(gd['costs'][13] / ci[13]))


straight drive iLQR costs [9000000.0, 0.10812, 0.10812, 0.10812] ; first control 0.1742
exact least effort for 3 m in 10 s: 12 d^2 / T^3 = 0.108 ; a at t = h/2: 0.174
iLQR iterations 99 ; cost after 0 and 13: 1422936.1483 0.2327 ; final 0.2172 ; 13 against final: 1.072
gradient descent after 13 and 1000 iterations: [174704.1, 2717.5] ; at 13 against iLQR at 13: 750669
